# OVRO-LWA source metacatalog

Identify sources in OVRO-LWA wide-field FITS images with **PyBDSF**, then fuse per-image catalogs into a **metacatalog** with one entry per unique sky position.

Each FITS file is associated with an **LST hour bin** (e.g. `01h`) and a **color band** (`Full`, `Red`, `Green`, or `Blue`). PyBDSF is run independently on every image.

**Workflow**
1. Discover FITS files and parse LST hour + band from filenames.
2. Run `bdsf.process_image` on each `(lst_hour, band)` image → per-image source catalogs.
3. **LST merge** (per band): cross-match detections across LST hours within each band. One row per source, covering the union of sky seen in any LST hour. Pick the detection whose peak flux is nearest the **median** flux over matching LST images; copy **all** properties from that row.
4. **Band merge** (sequential): cross-match Blue to Full → temporary metacatalog; then Green to that catalog; then Red. Each step uses beam-sized radii and median-flux picks. Unmatched sources from any band become their own row (`bands_present` lists contributing bands).

Set `REUSE_CACHED_CATALOGS = True` to load existing `sources_{lst}_{band}.csv` and/or `metacatalog_lst_{band}.csv` from `OUTPUT_DIR` instead of re-running PyBDSF or LST merge.

Reference conventions: `ovro-lwa-portal` (`fits_to_zarr_xradio.py`) and `image-plane-correction` (`source_detection.py`).

In [1]:
from __future__ import annotations

import os
import re
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Iterator

import astropy.units as u
import bdsf
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table

# --- user configuration ---------------------------------------------------
#FITS_ROOT = Path("/fast/claw")  # directory containing FITS images (searched recursively)
#OUTPUT_DIR = Path("/fast/claw/metacatalog")  # where catalogs are written
FITS_ROOT = Path("/Users/claw/data/lwa-metacatalog")  # directory containing FITS images (searched recursively)
OUTPUT_DIR = Path("/Users/claw/data/lwa-metacatalog/metacatalog")  # where catalogs are written

# PyBDSF detection parameters (see image-plane-correction/source_detection.py)
BDSF_KW = dict(
    thresh="hard",
    thresh_isl=7.0,
    thresh_pix=4.0,
    atrous_do=False,
    psf_vary_do=False,
    quiet=True,
    ncores=16,
)

# LST hour bins to process (must have Full + color-band FITS for each)
LST_HOURS = ["01h"]#, "02h", "03h"]
ASSOC_BANDS = ("Blue", "Green", "Red")

# When True, skip PyBDSF / LST merge if matching CSVs already exist in OUTPUT_DIR
REUSE_CACHED_CATALOGS = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Filename parsing

Generalized parsing supports several OVRO-LWA naming conventions:

| Convention | Example | LST hour | Band |
|------------|---------|----------|------|
| Deep color products | `I_01h_deep_Taper_R0_Full.fits` | `01h` | `Full` |
| LST color-band (portal ingest) | `Blue_I_10min_..._20250508_LST22h_t0001.fits` | `22h` | `Blue` |
| Parent directory | `.../01h/.../image.fits` | `01h` | from filename prefix |

Bands are one of `Full`, `Blue`, `Green`, `Red`. Additional LST hour bins and bands can be added without code changes as long as filenames follow these patterns.

In [2]:
COLOR_BANDS = ("Full", "Blue", "Green", "Red")
HOUR_DIR_RE = re.compile(r"^(\d{2})h$", re.IGNORECASE)

# Deep wideband color products: I_01h_deep_Taper_R0_Full.fits
DEEP_COLOR_RE = re.compile(
    r"^I_(\d{2})h_.*_(Full|Blue|Green|Red)\.fits$",
    re.IGNORECASE,
)

# Portal lst-color products: Blue_I_..._20250508_LST22h_t0001.fits
LST_COLOR_RE = re.compile(
    r"^(Full|Blue|Green|Red)_I_.*_(\d{8})_LST(\d{1,2})h_(t\d+)\.fits$",
    re.IGNORECASE,
)

# Band prefix fallback: Blue_I_....fits (LST from directory or LSTnnh elsewhere in name)
BAND_PREFIX_RE = re.compile(
    r"^(Full|Blue|Green|Red)_I_.*\.fits$",
    re.IGNORECASE,
)
LST_IN_NAME_RE = re.compile(r"LST(\d{1,2})h", re.IGNORECASE)


@dataclass(frozen=True, slots=True)
class FitsMetadata:
    path: Path
    lst_hour: str  # e.g. "01h"
    band: str  # Full | Blue | Green | Red
    time_key: str | None = None  # optional lst-color time bin key


def _format_lst_hour(hour: int | str) -> str:
    return f"{int(hour):02d}h"


def _lst_from_parents(path: Path) -> str | None:
    for parent in path.parents:
        m = HOUR_DIR_RE.match(parent.name)
        if m:
            return _format_lst_hour(m.group(1))
    return None


def parse_fits_metadata(path: Path) -> FitsMetadata | None:
    """Return LST hour and color band for a FITS path, or None if unrecognized."""
    name = path.name

    m = DEEP_COLOR_RE.match(name)
    if m:
        return FitsMetadata(path=path, lst_hour=_format_lst_hour(m.group(1)), band=m.group(2).title())

    m = LST_COLOR_RE.match(name)
    if m:
        band, ymd, lst_h, t_bin = m.group(1), m.group(2), m.group(3), m.group(4)
        return FitsMetadata(
            path=path,
            lst_hour=_format_lst_hour(lst_h),
            band=band.title(),
            time_key=f"{ymd}_LST{_format_lst_hour(lst_h)}_{t_bin}",
        )

    m = BAND_PREFIX_RE.match(name)
    if m:
        band = m.group(1).title()
        lst_h = None
        m_lst = LST_IN_NAME_RE.search(name)
        if m_lst:
            lst_h = _format_lst_hour(m_lst.group(1))
        else:
            lst_h = _lst_from_parents(path)
        if lst_h is None:
            return None
        return FitsMetadata(path=path, lst_hour=lst_h, band=band)

    return None


def discover_fits_files(root: Path, *, patterns: Iterable[str] = ("*.fits",)) -> list[FitsMetadata]:
    """Recursively find FITS files and parse metadata."""
    found: list[FitsMetadata] = []
    for pattern in patterns:
        for path in sorted(root.rglob(pattern)):
            meta = parse_fits_metadata(path)
            if meta is not None:
                found.append(meta)
    return found

def slot_glob_patterns(lst_hour: str, band: str) -> list[str]:
    patterns = [
        f"**/*_{lst_hour}_*_{band}.fits",
        f"**/I_{lst_hour}_*_{band}.fits",
        f"I_{lst_hour}_*_{band}.fits",
    ]
    if band != "Full":
        patterns.insert(0, f"**/{band}_I_*_LST{lst_hour[0:2]}h_*.fits")
        patterns.insert(1, f"**/{band}_I_*_LST{int(lst_hour[:-1])}h_*.fits")
    else:
        patterns.insert(0, f"**/Full_I_*_LST{lst_hour[0:2]}h_*.fits")
    return patterns


def resolve_fits_slot(root: Path, lst_hour: str, band: str) -> Path:
    """Resolve exactly one FITS path for an (lst_hour, band) slot."""
    for pattern in slot_glob_patterns(lst_hour, band):
        matches = sorted({p.resolve() for p in root.glob(pattern)})
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise FileNotFoundError(
                f"Ambiguous glob for ({lst_hour}, {band}): pattern {pattern!r} matched "
                f"{len(matches)} files: {[m.name for m in matches]}"
            )
    raise FileNotFoundError(f"No FITS found for ({lst_hour}, {band}) under {root}")


In [3]:
fits_files = discover_fits_files(FITS_ROOT)
summary = pd.DataFrame(
    {
        "path": [m.path.name for m in fits_files],
        "lst_hour": [m.lst_hour for m in fits_files],
        "band": [m.band for m in fits_files],
        "time_key": [m.time_key for m in fits_files],
    }
)
print(f"Found {len(fits_files)} FITS files under {FITS_ROOT}")
summary.sort_values(["lst_hour", "band"]).reset_index(drop=True)

Found 12 FITS files under /Users/claw/data/lwa-metacatalog


,path,lst_hour,band,time_key
0,I_01h_deep_Taper_R0_Blue.fits,01h,Blue,None
1,I_01h_deep_Taper_R0_Full.fits,01h,Full,None
2,I_01h_deep_Taper_R0_Green.fits,01h,Green,None
3,I_01h_deep_Taper_R0_Red.fits,01h,Red,None
4,I_02h_deep_Taper_R0_Blue.fits,02h,Blue,None
5,I_02h_deep_Taper_R0_Full.fits,02h,Full,None
6,I_02h_deep_Taper_R0_Green.fits,02h,Green,None
7,I_02h_deep_Taper_R0_Red.fits,02h,Red,None
8,I_03h_deep_Taper_R0_Blue.fits,03h,Blue,None
9,I_03h_deep_Taper_R0_Full.fits,03h,Full,None


## PyBDSF source detection

For each `(lst_hour, band)` image we:
1. Read the FITS primary HDU.
2. Sanitize non-finite pixels and ensure frequency keywords PyBDSF expects (`RESTFREQ`).
3. Pass the HDU directly to `bdsf.process_image`.
4. Export the Gaussian list (`catalog_type='gaul'`) as an Astropy `Table`, then convert to a pandas `DataFrame` with provenance columns.

In [4]:
GAUL_COLUMNS = [
    "RA",
    "DEC",
    "E_RA",
    "E_DEC",
    "Total_flux",
    "E_Total_flux",
    "Peak_flux",
    "E_Peak_flux",
    "Maj",
    "E_Maj",
    "Min",
    "E_Min",
    "PA",
    "E_PA",
    "DC_Maj",
    "DC_Min",
    "DC_PA",
    "S_Code",
    "Gaus_id",
    "Isl_id",
    "Source_id",
]


def _restfreq_hz(header: fits.Header) -> float | None:
    for key in ("RESTFREQ", "RESTFRQ", "CRVAL3", "FREQ"):
        if key in header:
            try:
                return float(header[key])
            except (TypeError, ValueError):
                continue
    return None


def _prepare_hdu(path: Path) -> fits.PrimaryHDU:
    """Read FITS, squeeze to 2D, sanitize NaNs, and fix header for PyBDSF."""
    with fits.open(path, memmap=True) as hdul:
        hdu = hdul[0]
        data = np.squeeze(np.asarray(hdu.data, dtype=np.float32))
        if data.ndim != 2:
            raise ValueError(f"Expected 2D image in {path.name}, got shape {data.shape}")
        data = np.where(np.isfinite(data), data, 0.0)
        header = hdu.header.copy()
    rf = _restfreq_hz(header)
    if rf is not None:
        header["RESTFREQ"] = rf
        header["RESTFRQ"] = rf
    return fits.PrimaryHDU(data=data, header=header)


def _beam_from_header(header: fits.Header) -> tuple[float, float, float]:
    if "BMAJ" not in header or "BMIN" not in header:
        raise ValueError("FITS header missing BMAJ/BMIN beam keywords")
    return (
        float(header["BMAJ"]),
        float(header["BMIN"]),
        float(header.get("BPA", 0.0)),
    )


def run_pybdsf_on_hdu(hdu: fits.PrimaryHDU, **process_kw) -> Table:
    """Run PyBDSF on an in-memory HDU and return the Gaussian catalog table."""
    beam = _beam_from_header(hdu.header)
    kw = dict(BDSF_KW)
    kw.update(process_kw)
    kw["beam"] = beam

    img = bdsf.process_image(hdu, **kw)

    with tempfile.NamedTemporaryFile(suffix=".gaul.fits", delete=False) as tmp:
        cat_path = tmp.name
    try:
        img.write_catalog(outfile=cat_path, format="fits", catalog_type="gaul", clobber=True)
        return Table.read(cat_path)
    finally:
        try:
            os.unlink(cat_path)
        except OSError:
            pass


def detect_sources(meta: FitsMetadata) -> pd.DataFrame:
    """Detect sources in one FITS image; return a catalog DataFrame."""
    hdu = _prepare_hdu(meta.path)
    bmaj, bmin, bpa = _beam_from_header(hdu.header)
    table = run_pybdsf_on_hdu(hdu)
    df = table.to_pandas()

    keep = [c for c in GAUL_COLUMNS if c in df.columns]
    df = df[keep].copy()
    df["lst_hour"] = meta.lst_hour
    df["band"] = meta.band
    df["source_file"] = meta.path.name
    if meta.time_key is not None:
        df["time_key"] = meta.time_key
    df["BMAJ"] = bmaj
    df["BMIN"] = bmin
    df["BPA"] = bpa
    return df


def sources_csv_path(lst_hour: str, band: str) -> Path:
    return OUTPUT_DIR / f"sources_{lst_hour}_{band}.csv"


def lst_merged_csv_path(band: str) -> Path:
    return OUTPUT_DIR / f"metacatalog_lst_{band}.csv"


def all_sources_cached() -> bool:
    return all(sources_csv_path(lst, band).is_file() for lst in LST_HOURS for band in COLOR_BANDS)


def all_lst_merged_cached() -> bool:
    return all(lst_merged_csv_path(band).is_file() for band in COLOR_BANDS)


def _ensure_bmaj_column(df: pd.DataFrame, lst_hour: str, band: str) -> pd.DataFrame:
    """Backfill BMAJ/BMIN/BPA from the FITS header when loading legacy source CSVs."""
    if "BMAJ" in df.columns and df["BMAJ"].notna().any():
        return df
    path = resolve_fits_slot(FITS_ROOT, lst_hour, band)
    with fits.open(path, memmap=True) as hdul:
        bmaj, bmin, bpa = _beam_from_header(hdul[0].header)
    out = df.copy()
    out["BMAJ"] = bmaj
    out["BMIN"] = bmin
    out["BPA"] = bpa
    return out


def load_sources_catalog(csv_path: Path, lst_hour: str, band: str) -> pd.DataFrame:
    """Load a per-image source catalog CSV, ensuring beam columns are present."""
    df = pd.read_csv(csv_path)
    df = _ensure_bmaj_column(df, lst_hour, band)
    return df


def load_per_image_catalogs_from_disk() -> dict[tuple[str, str], pd.DataFrame]:
    """Load all per-image catalogs from OUTPUT_DIR when every slot is cached."""
    catalogs: dict[tuple[str, str], pd.DataFrame] = {}
    for lst_hour in LST_HOURS:
        for band in COLOR_BANDS:
            csv_path = sources_csv_path(lst_hour, band)
            if not csv_path.is_file():
                raise FileNotFoundError(f"Missing cached catalog: {csv_path}")
            catalogs[(lst_hour, band)] = load_sources_catalog(csv_path, lst_hour, band)
    return catalogs


def load_lst_merged_from_disk() -> dict[str, pd.DataFrame]:
    """Load LST-merged per-band catalogs from OUTPUT_DIR."""
    merged: dict[str, pd.DataFrame] = {}
    for band in COLOR_BANDS:
        csv_path = lst_merged_csv_path(band)
        if not csv_path.is_file():
            raise FileNotFoundError(f"Missing cached LST merge: {csv_path}")
        merged[band] = pd.read_csv(csv_path)
    return merged


In [5]:
per_image_catalogs: dict[tuple[str, str], pd.DataFrame] = {}

for lst_hour in LST_HOURS:
    for band in COLOR_BANDS:
        key = (lst_hour, band)
        csv_path = sources_csv_path(lst_hour, band)

        if REUSE_CACHED_CATALOGS and csv_path.is_file():
            catalog = load_sources_catalog(csv_path, lst_hour, band)
            per_image_catalogs[key] = catalog
            print(f"Loaded: {csv_path.name}  (LST={lst_hour}, band={band}, n={len(catalog)})")
            continue

        path = resolve_fits_slot(FITS_ROOT, lst_hour, band)
        meta = parse_fits_metadata(path)
        if meta is None:
            meta = FitsMetadata(path=path, lst_hour=lst_hour, band=band)
        print(f"PyBDSF: {path.name}  (LST={lst_hour}, band={band})")
        catalog = detect_sources(meta)
        per_image_catalogs[key] = catalog
        catalog.to_csv(csv_path, index=False)
        print(f"  -> {len(catalog)} sources written to {csv_path}")

len(per_image_catalogs)

Loaded: sources_01h_Full.csv  (LST=01h, band=Full, n=2479)
Loaded: sources_01h_Blue.csv  (LST=01h, band=Blue, n=3843)
Loaded: sources_01h_Green.csv  (LST=01h, band=Green, n=2475)
Loaded: sources_01h_Red.csv  (LST=01h, band=Red, n=975)


4

## Metacatalog fusion

Two stages:

1. **`merge_lst_metacatalog`** — within each band, fuse detections from all LST hours. Matching uses greedy beam-sized clustering with vectorized `SkyCoord.separation`. The representative row is the detection whose `Peak_flux` is closest to the median over the cluster.

2. **`build_global_metacatalog`** — sequential band fusion with no Full master list: Blue↔Full → temp catalog, then Green→temp, then Red→temp. Each step uses `SkyCoord.search_around_sky` with per-pair beam radii. Unmatched sources append as new rows.

In [6]:
BAND_FIELDS = (
    "Peak_flux",
    "Total_flux",
    "RA",
    "DEC",
    "Maj",
    "Min",
    "PA",
    "DC_Maj",
    "DC_Min",
    "DC_PA",
)


def _pick_median_flux_row(df: pd.DataFrame, flux_col: str = "Peak_flux") -> pd.Series:
    """Return the row whose flux is closest to the median over *df*."""
    flux = df[flux_col].to_numpy(dtype=float)
    finite = np.isfinite(flux)
    if not finite.any():
        return df.iloc[0]
    med = float(np.nanmedian(flux[finite]))
    idx = int(np.nanargmin(np.abs(flux - med)))
    return df.iloc[idx]


def _skycoord_from_columns(df: pd.DataFrame) -> SkyCoord:
    return SkyCoord(
        ra=df["RA"].to_numpy(dtype=float) * u.deg,
        dec=df["DEC"].to_numpy(dtype=float) * u.deg,
    )


def _associate_catalogs(
    base_df: pd.DataFrame,
    band_df: pd.DataFrame,
) -> tuple[dict[int, list[int]], set[int]]:
    """Vectorized base↔band matching within per-pair beam radii.

    Returns ``{base_iloc: [band_iloc, ...]}`` and the set of matched band ilocs.
    """
    if base_df.empty or band_df.empty:
        return {}, set()

    base_sc = _skycoord_from_columns(base_df)
    band_sc = _skycoord_from_columns(band_df)
    bmaj_base = base_df["BMAJ"].to_numpy(dtype=float)
    bmaj_band = band_df["BMAJ"].to_numpy(dtype=float)

    search_radius = float(max(bmaj_base.max(), bmaj_band.max())) * u.deg
    # Astropy returns (idx_searcharound, idx_self) = (band, base)
    idx_band, idx_base, sep2d, _ = base_sc.search_around_sky(band_sc, search_radius)
    if len(idx_base) == 0:
        return {}, set()

    sep_deg = sep2d.to(u.deg).value
    limits = np.maximum(bmaj_base[idx_base], bmaj_band[idx_band])
    keep = sep_deg <= limits
    idx_base = idx_base[keep]
    idx_band = idx_band[keep]

    hits_by_base: dict[int, list[int]] = {}
    matched: set[int] = set()
    for i, j in zip(idx_base.tolist(), idx_band.tolist(), strict=True):
        hits_by_base.setdefault(i, []).append(j)
        matched.add(j)
    return hits_by_base, matched


def _cluster_by_sky_position(
    df: pd.DataFrame,
    *,
    sort_col: str = "Peak_flux",
    bmaj_col: str = "BMAJ",
) -> list[pd.DataFrame]:
    """Greedy beam-sized clustering; return one member DataFrame per cluster."""
    if df.empty:
        return []

    work = df[np.isfinite(df["RA"]) & np.isfinite(df["DEC"])].copy()
    work = work.sort_values(sort_col, ascending=False, na_position="last")

    cluster_ra: list[float] = []
    cluster_dec: list[float] = []
    cluster_bmaj: list[float] = []
    cluster_members: list[list[dict]] = []

    for row in work.itertuples(index=False):
        rd = row._asdict()
        ra_row = float(rd["RA"])
        dec_row = float(rd["DEC"])
        bmaj_row = float(rd.get(bmaj_col, np.nan))
        if not np.isfinite(bmaj_row):
            bmaj_row = 0.0

        best_idx: int | None = None
        if cluster_ra:
            sc = SkyCoord(ra=ra_row * u.deg, dec=dec_row * u.deg)
            cluster_sc = SkyCoord(
                ra=np.asarray(cluster_ra, dtype=float) * u.deg,
                dec=np.asarray(cluster_dec, dtype=float) * u.deg,
            )
            seps = sc.separation(cluster_sc).deg
            radii = np.maximum(bmaj_row, np.asarray(cluster_bmaj, dtype=float))
            within = seps <= radii
            if within.any():
                candidates = np.where(within)[0]
                best_idx = int(candidates[np.argmin(seps[candidates])])

        if best_idx is None:
            cluster_ra.append(ra_row)
            cluster_dec.append(dec_row)
            cluster_bmaj.append(bmaj_row)
            cluster_members.append([rd])
        else:
            cluster_members[best_idx].append(rd)
            rep = _pick_median_flux_row(pd.DataFrame(cluster_members[best_idx]))
            cluster_ra[best_idx] = float(rep["RA"])
            cluster_dec[best_idx] = float(rep["DEC"])
            cluster_bmaj[best_idx] = max(cluster_bmaj[best_idx], bmaj_row)

    return [pd.DataFrame(members) for members in cluster_members]


def merge_lst_metacatalog(catalogs: Iterable[pd.DataFrame], *, band: str) -> pd.DataFrame:
    """Fuse per-LST detections within one band → one row per source."""
    combined = pd.concat(list(catalogs), ignore_index=True)
    if combined.empty:
        return pd.DataFrame()

    rows: list[dict] = []
    for members in _cluster_by_sky_position(combined):
        rep = _pick_median_flux_row(members)
        entry = rep.to_dict()
        entry["band"] = band
        entry["n_lst_contributions"] = len(members)
        entry["lst_hours"] = ",".join(sorted(members["lst_hour"].unique()))
        entry["representative_lst"] = rep["lst_hour"]
        rows.append(entry)

    meta = pd.DataFrame(rows)
    return meta.sort_values("Peak_flux", ascending=False, na_position="last").reset_index(drop=True)


def _empty_band_cols() -> dict:
    out: dict = {}
    for band in ASSOC_BANDS:
        for field in BAND_FIELDS:
            out[f"{field}_{band}"] = np.nan
        out[f"n_assoc_{band}"] = 0
    return out


def _lst_meta_from_band_row(row: pd.Series) -> tuple[int, str, str]:
    return (
        int(row.get("n_lst_contributions", 1)),
        str(row.get("lst_hours", row.get("lst_hour", ""))),
        str(row.get("representative_lst", row.get("lst_hour", ""))),
    )


def _primary_fields_from_band(row: pd.Series) -> dict:
    return {
        "RA": row["RA"],
        "DEC": row["DEC"],
        "Peak_flux": row["Peak_flux"],
        "Total_flux": row["Total_flux"],
        "Maj": row["Maj"],
        "Min": row["Min"],
        "PA": row["PA"],
        "DC_Maj": row.get("DC_Maj", np.nan),
        "DC_Min": row.get("DC_Min", np.nan),
        "DC_PA": row.get("DC_PA", np.nan),
    }


def _attach_band_columns(entry: dict, band_row: pd.Series, band: str, n_assoc: int) -> None:
    entry[f"n_assoc_{band}"] = n_assoc
    for field in BAND_FIELDS:
        entry[f"{field}_{band}"] = band_row[field]
    entry[f"source_file_{band}"] = band_row.get("source_file", "")


def _update_bands_present(entry: dict, *bands: str) -> None:
    present = {b for b in str(entry.get("bands_present", "")).split(",") if b}
    present.update(bands)
    order = ("Full", "Blue", "Green", "Red")
    entry["bands_present"] = ",".join(b for b in order if b in present)


def _seed_row_from_band(band_row: pd.Series, band: str) -> dict:
    """One metacatalog row seeded from a single-band LST-merged detection."""
    n_lst, lst_hours, rep_lst = _lst_meta_from_band_row(band_row)
    entry = {
        "origin_band": band,
        "bands_present": band,
        **_primary_fields_from_band(band_row),
        "BMAJ_match": float(band_row["BMAJ"]),
        "n_lst_contributions": n_lst,
        "lst_hours": lst_hours,
        "representative_lst": rep_lst,
    }
    entry.update(_empty_band_cols())
    if band == "Full":
        entry["BMAJ_full"] = float(band_row["BMAJ"])
        entry["source_file_Full"] = band_row.get("source_file", "")
    else:
        entry["BMAJ_full"] = np.nan
        _attach_band_columns(entry, band_row, band, 1)
    return entry


def merge_full_and_blue(full_df: pd.DataFrame, blue_df: pd.DataFrame) -> pd.DataFrame:
    """Cross-match Blue onto Full; one row per deduplicated sky position."""
    full_df = full_df.reset_index(drop=True)
    blue_df = blue_df.reset_index(drop=True)
    hits, matched_blue = _associate_catalogs(full_df, blue_df)

    rows: list[dict] = []
    for i, frow in full_df.iterrows():
        entry = _seed_row_from_band(frow, "Full")
        blues = hits.get(i, [])
        if blues:
            sub = blue_df.iloc[blues]
            best = _pick_median_flux_row(sub)
            _attach_band_columns(entry, best, "Blue", len(blues))
            _update_bands_present(entry, "Full", "Blue")
            entry["BMAJ_match"] = max(float(entry["BMAJ_match"]), float(best["BMAJ"]))
        rows.append(entry)

    for j, brow in blue_df.iterrows():
        if j not in matched_blue:
            rows.append(_seed_row_from_band(brow, "Blue"))
    return pd.DataFrame(rows)


def associate_band_into_metacatalog(
    meta_df: pd.DataFrame,
    band_df: pd.DataFrame,
    band: str,
) -> pd.DataFrame:
    """Cross-match one color band onto the current metacatalog; append unmatched band rows."""
    band_df = band_df.reset_index(drop=True)
    if meta_df.empty:
        return pd.DataFrame([_seed_row_from_band(brow, band) for _, brow in band_df.iterrows()])

    meta_df = meta_df.reset_index(drop=True)
    match_base = meta_df[["RA", "DEC"]].copy()
    match_base["BMAJ"] = meta_df["BMAJ_match"].to_numpy(dtype=float)
    hits, matched_band = _associate_catalogs(match_base, band_df)

    rows: list[dict] = []
    for i, mrow in meta_df.iterrows():
        entry = mrow.to_dict()
        band_hits = hits.get(i, [])
        if band_hits:
            sub = band_df.iloc[band_hits]
            best = _pick_median_flux_row(sub)
            _attach_band_columns(entry, best, band, len(band_hits))
            _update_bands_present(entry, band)
            entry["BMAJ_match"] = max(float(entry["BMAJ_match"]), float(best["BMAJ"]))
        rows.append(entry)

    for j, brow in band_df.iterrows():
        if j not in matched_band:
            rows.append(_seed_row_from_band(brow, band))
    return pd.DataFrame(rows)


def build_global_metacatalog(lst_merged: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Fuse LST-merged per-band catalogs via sequential cross-matching."""
    temp = merge_full_and_blue(lst_merged["Full"], lst_merged["Blue"])
    for band in ("Green", "Red"):
        temp = associate_band_into_metacatalog(temp, lst_merged[band], band)
    meta = temp
    meta.insert(0, "meta_id", range(len(meta)))
    return meta.sort_values("Peak_flux", ascending=False, na_position="last").reset_index(drop=True)

In [7]:
lst_merged: dict[str, pd.DataFrame] = {}

if REUSE_CACHED_CATALOGS and all_lst_merged_cached():
    lst_merged = load_lst_merged_from_disk()
    for band in COLOR_BANDS:
        print(
            f"LST merge ({band}): loaded {len(lst_merged[band])} sources from {lst_merged_csv_path(band)}"
        )
else:
    if not per_image_catalogs and REUSE_CACHED_CATALOGS and all_sources_cached():
        per_image_catalogs.update(load_per_image_catalogs_from_disk())
        print(f"Loaded {len(per_image_catalogs)} per-image catalogs from {OUTPUT_DIR}")

    for band in COLOR_BANDS:
        band_catalogs = [
            per_image_catalogs[(lst, band)]
            for lst in LST_HOURS
            if (lst, band) in per_image_catalogs
        ]
        merged = merge_lst_metacatalog(band_catalogs, band=band)
        lst_merged[band] = merged
        out_csv = lst_merged_csv_path(band)
        merged.to_csv(out_csv, index=False)
        print(f"LST merge ({band}): {len(merged)} sources -> {out_csv}")

metacatalog = build_global_metacatalog(lst_merged)

meta_csv = OUTPUT_DIR / "metacatalog.csv"
meta_fits = OUTPUT_DIR / "metacatalog.fits"
metacatalog.to_csv(meta_csv, index=False)
Table.from_pandas(metacatalog).write(meta_fits, overwrite=True)

if per_image_catalogs:
    n_inputs = sum(len(df) for df in per_image_catalogs.values())
    input_desc = f"{n_inputs} per-image detections"
else:
    n_inputs = sum(int(df["n_lst_contributions"].sum()) for df in lst_merged.values())
    input_desc = f"{n_inputs} LST-merged rows (cached)"
print(f"\nGlobal metacatalog: {len(metacatalog)} sources from {input_desc}")
print(f"Wrote {meta_csv}")
print(f"Wrote {meta_fits}")
metacatalog.head(10)

LST merge (Full): loaded 2921 sources from /Users/claw/data/lwa-metacatalog/metacatalog/metacatalog_lst_Full.csv
LST merge (Blue): loaded 4952 sources from /Users/claw/data/lwa-metacatalog/metacatalog/metacatalog_lst_Blue.csv
LST merge (Green): loaded 2773 sources from /Users/claw/data/lwa-metacatalog/metacatalog/metacatalog_lst_Green.csv
LST merge (Red): loaded 1101 sources from /Users/claw/data/lwa-metacatalog/metacatalog/metacatalog_lst_Red.csv

Global metacatalog: 4944 sources from 9772 per-image detections
Wrote /Users/claw/data/lwa-metacatalog/metacatalog/metacatalog.csv
Wrote /Users/claw/data/lwa-metacatalog/metacatalog/metacatalog.fits


,meta_id,origin_band,bands_present,RA,DEC,Peak_flux,Total_flux,Maj,Min,PA,...,PA_Red,DC_Maj_Red,DC_Min_Red,DC_PA_Red,n_assoc_Red,BMAJ_full,source_file_Full,source_file_Blue,source_file_Green,source_file_Red
0,0,Full,"Full,Blue,Green,Red",76.171346,38.108672,133.635365,132.253380,0.286102,0.259473,78.343485,...,70.416445,0.0,0.0,0.0,1,0.271752,I_01h_deep_Taper_R0_Full.fits,I_01h_deep_Taper_R0_Blue.fits,I_02h_deep_Taper_R0_Green.fits,I_02h_deep_Taper_R0_Red.fits
1,1,Full,"Full,Blue,Green,Red",49.947786,41.507587,119.370788,60.571320,0.193877,0.152462,65.724720,...,44.433169,0.0,0.0,0.0,1,0.271752,I_01h_deep_Taper_R0_Full.fits,I_01h_deep_Taper_R0_Blue.fits,I_02h_deep_Taper_R0_Green.fits,I_03h_deep_Taper_R0_Red.fits
2,2,Full,"Full,Blue,Green,Red",64.592951,38.037622,118.048747,111.850921,0.275708,0.224309,68.619080,...,62.012227,0.0,0.0,0.0,1,0.271752,I_01h_deep_Taper_R0_Full.fits,I_02h_deep_Taper_R0_Blue.fits,I_01h_deep_Taper_R0_Green.fits,I_02h_deep_Taper_R0_Red.fits
3,3,Full,"Full,Blue,Green,Red",69.276286,29.666999,112.858216,124.270613,0.324915,0.251006,54.271922,...,63.073934,0.0,0.0,0.0,1,0.282342,I_02h_deep_Taper_R0_Full.fits,I_03h_deep_Taper_R0_Blue.fits,I_01h_deep_Taper_R0_Green.fits,I_02h_deep_Taper_R0_Red.fits
4,4,Full,"Full,Blue,Green,Red",6.339984,64.138927,81.350899,33.271646,0.219100,0.146359,23.837481,...,14.921531,0.0,0.0,0.0,1,0.282342,I_02h_deep_Taper_R0_Full.fits,I_03h_deep_Taper_R0_Blue.fits,I_02h_deep_Taper_R0_Green.fits,I_02h_deep_Taper_R0_Red.fits
5,5,Full,"Full,Blue,Green,Red",10.785425,52.053865,60.522198,41.526236,0.285983,0.184953,15.066310,...,25.740218,0.0,0.0,0.0,1,0.281562,I_03h_deep_Taper_R0_Full.fits,I_02h_deep_Taper_R0_Blue.fits,I_02h_deep_Taper_R0_Green.fits,I_02h_deep_Taper_R0_Red.fits
6,6,Full,"Full,Blue,Green,Red",29.293061,28.857535,54.733421,49.761140,0.254500,0.196244,46.336761,...,47.212475,0.0,0.0,0.0,1,0.271752,I_01h_deep_Taper_R0_Full.fits,I_01h_deep_Taper_R0_Blue.fits,I_01h_deep_Taper_R0_Green.fits,I_01h_deep_Taper_R0_Red.fits
7,7,Full,"Full,Blue,Green,Red",17.219542,13.325362,54.228265,25.706695,0.210826,0.190857,34.821129,...,38.199738,0.0,0.0,0.0,1,0.281562,I_03h_deep_Taper_R0_Full.fits,I_02h_deep_Taper_R0_Blue.fits,I_02h_deep_Taper_R0_Green.fits,I_02h_deep_Taper_R0_Red.fits
8,8,Full,"Full,Blue,Green,Red",24.104712,20.954548,50.910373,24.778808,0.211159,0.172679,35.952577,...,38.185627,0.0,0.0,0.0,1,0.281562,I_03h_deep_Taper_R0_Full.fits,I_02h_deep_Taper_R0_Blue.fits,I_02h_deep_Taper_R0_Green.fits,I_02h_deep_Taper_R0_Red.fits
9,9,Full,"Full,Blue,Green,Red",24.425780,33.164049,49.114238,20.530857,0.197691,0.149943,31.807467,...,45.148946,0.0,0.0,0.0,1,0.281562,I_03h_deep_Taper_R0_Full.fits,I_02h_deep_Taper_R0_Blue.fits,I_02h_deep_Taper_R0_Green.fits,I_01h_deep_Taper_R0_Red.fits


In [8]:
# LST merge yield (Full band)
full_lst = lst_merged["Full"]
multi_lst = full_lst[full_lst["n_lst_contributions"] > 1].sort_values("n_lst_contributions", ascending=False)
print(f"Full-band sources after LST merge: {len(full_lst)}")
print(f"  seen in multiple LST hours: {len(multi_lst)}")
if len(multi_lst):
    display(multi_lst.head(10)[["RA", "DEC", "Peak_flux", "n_lst_contributions", "lst_hours", "representative_lst"]])

# Global band merge
print(f"\nGlobal rows by bands_present (top 10):")
print(metacatalog["bands_present"].value_counts().head(10))

multi_band = metacatalog[metacatalog["bands_present"].str.contains(",")]
print(f"Rows with multiple bands: {len(multi_band)}")
metacatalog.head(10)[["meta_id", "RA", "DEC", "bands_present", "origin_band", "Peak_flux", "lst_hours", "n_assoc_Blue", "n_assoc_Green", "n_assoc_Red"]]

Full-band sources after LST merge: 2921
  seen in multiple LST hours: 2083


,RA,DEC,Peak_flux,n_lst_contributions,lst_hours,representative_lst
52,83.463512,22.139776,27.792369,15,"01h,02h,03h",01h
1,49.947786,41.507587,119.370788,10,"01h,02h,03h",01h
3,69.276286,29.666999,112.858216,10,"01h,02h,03h",02h
242,52.044441,55.150278,13.174132,9,"01h,02h,03h",02h
492,16.787899,32.462786,9.130606,8,"01h,02h,03h",02h
55,49.561908,41.905744,27.660132,7,"01h,02h,03h",02h
51,72.451332,45.037292,27.877489,7,"01h,02h,03h",03h
24,300.145930,40.843282,34.548697,7,"01h,02h,03h",01h
4,6.339984,64.138927,81.350899,7,"01h,02h,03h",02h
2622,60.102037,40.072223,3.927900,6,"01h,02h,03h",03h



Global rows by bands_present (top 10):
bands_present
Blue                   1706
Full,Blue,Green        1359
Full,Blue,Green,Red    1171
Full,Blue               305
Blue,Green              194
Green                    45
Blue,Red                 40
Full                     30
Blue,Green,Red           26
Full,Blue,Red            22
Name: count, dtype: int64
Rows with multiple bands: 3157


,meta_id,RA,DEC,bands_present,origin_band,Peak_flux,lst_hours,n_assoc_Blue,n_assoc_Green,n_assoc_Red
0,0,76.171346,38.108672,"Full,Blue,Green,Red",Full,133.635365,"01h,02h,03h",2,1,1
1,1,49.947786,41.507587,"Full,Blue,Green,Red",Full,119.370788,"01h,02h,03h",1,1,1
2,2,64.592951,38.037622,"Full,Blue,Green,Red",Full,118.048747,"01h,02h,03h",1,1,1
3,3,69.276286,29.666999,"Full,Blue,Green,Red",Full,112.858216,"01h,02h,03h",2,1,1
4,4,6.339984,64.138927,"Full,Blue,Green,Red",Full,81.350899,"01h,02h,03h",1,1,1
5,5,10.785425,52.053865,"Full,Blue,Green,Red",Full,60.522198,"01h,02h,03h",1,1,1
6,6,29.293061,28.857535,"Full,Blue,Green,Red",Full,54.733421,"01h,02h,03h",1,1,1
7,7,17.219542,13.325362,"Full,Blue,Green,Red",Full,54.228265,"01h,02h,03h",1,1,1
8,8,24.104712,20.954548,"Full,Blue,Green,Red",Full,50.910373,"01h,02h,03h",1,1,1
9,9,24.425780,33.164049,"Full,Blue,Green,Red",Full,49.114238,"01h,02h,03h",1,1,1
